# Modelo de classificação

Reconstruído em 28/10/2026 a partir da [conversa compartilhada](https://chatgpt.com/share/6ae27680-e994-83e9-a410-8026e2a56e00) e do texto fornecido. Os arquivos originais tinham 0 bytes. A integração dos fatores SVD com a rede não havia sido definida na conversa; por isso, este notebook contém as funções recuperadas, sem executar treinamento.

**Pendências metodológicas:** representação de entrada (B e C ou A_k), ativação, codificação da saída, divisão dos dados, tamanho do lote, dimensões de entrada e saída, número de neurônios por camada, taxa de aprendizado e épocas.


In [ ]:
import numpy as np
from PIL import Image
from scipy.sparse.linalg import svds


def transformar(A, k, escala_cinza=True):
    """Fatora uma imagem em B_k C_k por canal.

    Em escala de cinza, retorna (B_k, C_k). Em RGB, retorna
    (B_R, C_R, B_G, C_G, B_B, C_B), nessa ordem.
    """
    imagem = np.asarray(A)

    if not isinstance(escala_cinza, (bool, np.bool_)):
        raise TypeError("escala_cinza deve ser True ou False.")

    def fatorar_canal(canal):
        canal = np.asarray(canal, dtype=float)
        if canal.ndim != 2:
            raise ValueError("Cada canal deve ser uma matriz bidimensional.")
        if not isinstance(k, (int, np.integer)) or not 1 <= k < min(canal.shape):
            raise ValueError(f"k deve ser inteiro entre 1 e {min(canal.shape) - 1}.")

        U_k, sigma_k, V_k_T = svds(canal, k=k, which="LM")
        ordem_decrescente = np.argsort(sigma_k)[::-1]
        U_k = U_k[:, ordem_decrescente]
        sigma_k = sigma_k[ordem_decrescente]
        V_k_T = V_k_T[ordem_decrescente, :]

        B_k = U_k
        C_k = sigma_k[:, None] * V_k_T
        return B_k, C_k

    if escala_cinza:
        if imagem.ndim == 3 and imagem.shape[2] == 3:
            if imagem.dtype != np.uint8:
                raise TypeError("Para converter RGB em cinza, A deve ter pixels uint8.")
            canal_cinza = np.asarray(Image.fromarray(imagem).convert("L"))
        elif imagem.ndim == 2:
            canal_cinza = imagem
        else:
            raise ValueError("A deve ser uma matriz 2D ou uma imagem RGB (altura, largura, 3).")
        return fatorar_canal(canal_cinza)

    if imagem.ndim != 3 or imagem.shape[2] != 3:
        raise ValueError("Com escala_cinza=False, A deve ter formato (altura, largura, 3).")

    B_R, C_R = fatorar_canal(imagem[:, :, 0])
    B_G, C_G = fatorar_canal(imagem[:, :, 1])
    B_B, C_B = fatorar_canal(imagem[:, :, 2])
    return B_R, C_R, B_G, C_G, B_B, C_B

### Retorno de `transformar`

- `escala_cinza=True`: `B_k, C_k`, com `A_k = B_k @ C_k`. Uma entrada RGB é convertida para escala de cinza; uma matriz 2D é usada diretamente.
- `escala_cinza=False`: `B_R, C_R, B_G, C_G, B_B, C_B`, correspondentes aos canais vermelho, verde e azul. Cada aproximação é `A_{k,c} = B_c @ C_c`, para `c` em `{R, G, B}`.

A saída RGB contém seis matrizes separadas, na ordem pedida: `B_1, C_1, B_2, C_2, B_3, C_3`, com índices `1=R`, `2=G` e `3=B`.


## PyTorch

A conversa incluiu a instalação com `%pip install --user torch torchvision` para um ambiente compartilhado. Execute essa instalação separadamente somente se for necessária no ambiente em que abrir o notebook.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
class RedeNeural(nn.Module):
    def __init__(self, entrada_dim, camadas_ocultas, saida_dim, ativacao):
        super().__init__()
        if len(camadas_ocultas) != 10:
            raise ValueError("A rede deve ter exatamente 10 camadas ocultas.")
        if any(n <= 0 for n in camadas_ocultas):
            raise ValueError("Cada camada deve ter pelo menos um neurônio.")
        self.camadas = nn.ModuleList()
        dimensoes = [entrada_dim] + camadas_ocultas
        for i in range(10):
            self.camadas.append(nn.Linear(dimensoes[i], dimensoes[i + 1]))
        self.saida = nn.Linear(camadas_ocultas[-1], saida_dim)
        self.ativacao = ativacao

    def forward(self, x):
        for camada in self.camadas:
            x = self.ativacao(camada(x))
        return self.saida(x)

In [ ]:
camadas_ocultas = [
    None,  # Camada 1
    None,  # Camada 2
    None,  # Camada 3
    None,  # Camada 4
    None,  # Camada 5
    None,  # Camada 6
    None,  # Camada 7
    None,  # Camada 8
    None,  # Camada 9
    None   # Camada 10
]
taxa_aprendizado = None
epocas = None

In [ ]:
def treinar(modelo, dataloader, epocas, taxa_aprendizado):
    criterio = nn.MSELoss()
    otimizador = optim.Adam(modelo.parameters(), lr=taxa_aprendizado)
    historico = []
    for epoca in range(epocas):
        modelo.train()
        perda_total = 0
        total_amostras = 0
        for X, y in dataloader:
            otimizador.zero_grad()
            previsao = modelo(X)
            perda = criterio(previsao, y)
            perda.backward()
            otimizador.step()
            quantidade = X.shape[0]
            perda_total += perda.item() * quantidade
            total_amostras += quantidade
        mse_medio = perda_total / total_amostras
        historico.append(mse_medio)
        print(f"Época {epoca + 1}/{epocas} | MSE = {mse_medio:.6f}")
    return historico

In [ ]:
def plotar_treinamento(historico):
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(historico) + 1), historico)
    plt.xlabel("Época")
    plt.ylabel("MSE")
    plt.title("Evolução do erro durante o treinamento")
    plt.grid(True)
    plt.tight_layout()
    plt.show()